In [1]:
import sys
from pathlib import Path

# Ajusta o path para a pasta src (projeto/notebooks -> projeto/src)
sys.path.append(str((Path.cwd().parent / "src").resolve()))

from _01_list_extract import list_extract
from _02_discourses_extract import discourses_extract
from _03_discourse_preprocessing import preprocessing
from _04_embeddings import generate_discourse_embeddings
# from _04_topics import topics_main
# from _05_llm_analysis import llm_analysis

import pandas as pd
from gensim.models import LdaModel

import os
import shutil

In [ ]:
# teste de extração
# list_extract("02/07/2022", "05/07/2022")

.....................................
... Função list_extract iniciada! ...
... URL base:  https://www.camara.leg.br/internet/sitaqweb/resultadoPesquisaDiscursos.asp?txIndexacao=&CurrentPage=1&BasePesq=plenario&txOrador=&txPartido=&dtInicio=02/07/2022&dtFim=05/07/2022&txUF=&txSessao=&listaTipoSessao=&listaTipoInterv=&inFalaPres=&listaTipoFala=&listaFaseSessao=&txAparteante=&listaEtapa=&CampoOrdenacao=dtSessao&TipoOrdenacao=DESC&PageSize=550&txTexto=&txSumario= ...
... Acessando página! ...
... Status: 200 ...
... Sucesso!!! ...
... O número total de discursos encontrados é: 158 ...
... O número total de páginas é: 1 ...
... Formatando dados extraídos! ...
... Salvando as informações! ...
... Função list_extract encerrada! ...
......................................


(           parlamentar    partido uf_partido      sessao                 fase  \
 0            LUIZ LIMA         PL         RJ    105.2022         ENCERRAMENTO   
 1      ALENCAR SANTANA         PT         SP    105.2022  BREVES COMUNICAÇÕES   
 2       NEUCIMAR FRAGA         PP         ES    105.2022  BREVES COMUNICAÇÕES   
 3       CARMEN ZANOTTO  CIDADANIA         SC    105.2022  BREVES COMUNICAÇÕES   
 4    BENEDITA DA SILVA         PT         RJ    105.2022  BREVES COMUNICAÇÕES   
 ..                 ...        ...        ...         ...                  ...   
 153    AFONSO FLORENCE         PT         BA  021.4.56.N         ORDEM DO DIA   
 154    AFONSO FLORENCE         PT         BA  021.4.56.N         ORDEM DO DIA   
 155    AFONSO FLORENCE         PT         BA  021.4.56.N         ORDEM DO DIA   
 156    AFONSO FLORENCE         PT         BA  021.4.56.N         ORDEM DO DIA   
 157    AFONSO FLORENCE         PT         BA  021.4.56.N         ORDEM DO DIA   
 
            da

In [ ]:
# discursos antes da eleição
# df, name = preprocessing ( *discourses_extract ( *list_extract("02/07/2022", "29/10/2022") ) )

.....................................
... Função list_extract iniciada! ...
... URL base:  https://www.camara.leg.br/internet/sitaqweb/resultadoPesquisaDiscursos.asp?txIndexacao=&CurrentPage=1&BasePesq=plenario&txOrador=&txPartido=&dtInicio=02/07/2022&dtFim=29/10/2022&txUF=&txSessao=&listaTipoSessao=&listaTipoInterv=&inFalaPres=&listaTipoFala=&listaFaseSessao=&txAparteante=&listaEtapa=&CampoOrdenacao=dtSessao&TipoOrdenacao=DESC&PageSize=550&txTexto=&txSumario= ...
... Acessando página! ...
... Status: 200 ...
... Sucesso!!! ...
... O número total de discursos encontrados é: 2329 ...
... O número total de páginas é: 5 ...
... Acessando páginas adicionais! ...
... Adicionando os discursos da página: ... 2
... Adicionando os discursos da página: ... 3
... Adicionando os discursos da página: ... 4
... Adicionando os discursos da página: ... 5
... Formatando dados extraídos! ...
... Salvando as informações! ...
... Função list_extract encerrada! ...
......................................
..

In [ ]:
# discursos depois da eleição
# df, name = preprocessing ( *discourses_extract ( *list_extract("31/10/2022", "02/01/2023") ) )

.....................................
... Função list_extract iniciada! ...
... URL base:  https://www.camara.leg.br/internet/sitaqweb/resultadoPesquisaDiscursos.asp?txIndexacao=&CurrentPage=1&BasePesq=plenario&txOrador=&txPartido=&dtInicio=31/10/2022&dtFim=02/01/2023&txUF=&txSessao=&listaTipoSessao=&listaTipoInterv=&inFalaPres=&listaTipoFala=&listaFaseSessao=&txAparteante=&listaEtapa=&CampoOrdenacao=dtSessao&TipoOrdenacao=DESC&PageSize=550&txTexto=&txSumario= ...
... Acessando página! ...
... Status: 200 ...
... Sucesso!!! ...
... O número total de discursos encontrados é: 2401 ...
... O número total de páginas é: 5 ...
... Acessando páginas adicionais! ...
... Adicionando os discursos da página: ... 2
... Adicionando os discursos da página: ... 3
... Adicionando os discursos da página: ... 4
... Adicionando os discursos da página: ... 5
... Formatando dados extraídos! ...
... Salvando as informações! ...
... Função list_extract encerrada! ...
......................................
..

## Testes para avaliação do threshold do parsing semântico

In [ ]:
# Teste com discursos reais do CSV para ajustar o threshold + inspeção manual de chunks
from pathlib import Path

csv_path = Path.cwd().parent / "data" / "running_files" / "political_discourses_ini_31102022_fim_02012023.csv"
df_csv = pd.read_csv(csv_path)

# Mantém apenas discursos com texto disponível
df_csv = df_csv[df_csv["preprocess_disc"].notna()].copy()
df_csv = df_csv[df_csv["partido"].notna()].copy()

# Seleciona alguns discursos para inspeção rápida (2 por partido, até 6 no total)
sample_df = df_csv.groupby("partido", group_keys=False).head(2).head(6).copy()
print("Partidos no teste:", sorted(sample_df["partido"].astype(str).unique().tolist()))
print("Discursos no teste:", len(sample_df))

thresholds = [0.35, 0.45, 0.55]
results = []
inspection_data = {}  # guarda dados para inspeção manual por threshold

for threshold in thresholds:
    emb_df_test, emb_matrix_test, _ = generate_discourse_embeddings(
        dataframe=sample_df,
        party=None,
        text_col="raw_disc",
        similarity_threshold=threshold,
        min_sentences_per_chunk=1,
        max_sentences_per_chunk=None,
        save_files=False,
    )

    # Agrupa chunks por discurso para facilitar inspeção manual
    inspect_df = (
        emb_df_test
        .sort_values(["source_index", "chunk_id"])
        .groupby(["source_index", "partido", "parlamentar"], as_index=False)["chunk_text"]
        .apply(list)
        .rename(columns={"chunk_text": "chunks"})
    )
    inspect_df["n_chunks"] = inspect_df["chunks"].apply(len)

    inspection_data[threshold] = {
        "emb_df": emb_df_test,
        "emb_matrix": emb_matrix_test,
        "inspect_df": inspect_df,
    }

    results.append({
        "threshold": threshold,
        "chunks_total": len(emb_df_test),
        "embedding_shape": emb_matrix_test.shape,
        "media_chunks_por_discurso": float(inspect_df["n_chunks"].mean()) if not inspect_df.empty else 0.0,
    })

    print("\nThreshold:", threshold)
    print("Quantidade total de chunks:", len(emb_df_test))
    print("Dimensão da matriz de embeddings:", emb_matrix_test.shape)
    display(inspect_df[["source_index", "partido", "parlamentar", "n_chunks"]])

# Resumo comparativo
results_df = pd.DataFrame(results)
display(results_df)

Partidos no teste: ['NOVO', 'PCdoB', 'PL', 'PT']
Discursos no teste: 6
...................................................
... Função generate_discourse_embeddings iniciada ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1500.99it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... Total de chunks gerados: 58 ...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.42it/s]

... Função generate_discourse_embeddings encerrada ...
....................................................

Threshold: 0.35
Quantidade total de chunks: 58
Dimensão da matriz de embeddings: (58, 384)


...

,source_index,partido,parlamentar,n_chunks
0,0,NOVO,Marcel Van Hattem,18
1,1,PCdoB,Renildo Calheiros,15
2,2,PT,Reginaldo Lopes,8
3,3,NOVO,Marcel Van Hattem,11
4,4,PL,Bia Kicis,28
5,6,PT,Reginaldo Lopes,36


,threshold,chunks_total,embedding_shape,media_chunks_por_discurso
0,0.35,58,"(58, 384)",9.666667
1,0.45,91,"(91, 384)",15.166667
2,0.55,116,"(116, 384)",19.333333



Inspeção manual para threshold: 0.45


,source_index,partido,parlamentar,n_chunks
0,0,NOVO,Marcel Van Hattem,10
1,1,PCdoB,Renildo Calheiros,12
2,2,PT,Reginaldo Lopes,7
3,3,NOVO,Marcel Van Hattem,8
4,4,PL,Bia Kicis,26
5,6,PT,Reginaldo Lopes,28



Discurso selecionado:
source_index: 0
partido: NOVO
parlamentar: Marcel Van Hattem
n_chunks: 10

--- Chunk 1 ---
O SR. MARCEL VAN HATTEM (NOVO - RS. Pela ordem. Sem revisão do orador.) - Sra. Presidente, caros colegas Parlamentares, este meu discurso aqui na Câmara dos Deputados, nesta noite, talvez o último discurso desta legislatura, afinal de contas faltam apenas 6 minutos para que encerre o dia e assim também se encerre esta legislatura antes do recesso, é apenas para lembrar que fiz uma questão de ordem ao Presidente da Câmara, Deputado Arthur Lira.

--- Chunk 2 ---
Soube, por meio do Presidente, que ele já tinha a decisão lavrada para ser lida quando questionado.

--- Chunk 3 ---
Infelizmente ele não está neste momento aqui presidindo a sessão, a Deputada Bia Kicis tem feito isso com louvor, mas, obviamente, na ausência do Deputado Arthur Lira, não faço à Deputada inquirição sobre a questão de ordem em relação à instalação da CPI que trata do abuso de autoridade dos Ministros do

In [7]:
results_df

,threshold,chunks,embedding_shape
0,0.35,58,"(58, 384)"
1,0.45,91,"(91, 384)"
2,0.55,116,"(116, 384)"


In [17]:

# ============================
# INSPEÇÃO MANUAL (ajuste aqui)
# ============================
threshold_to_inspect = 0.35

inspect_df = inspection_data[threshold_to_inspect]["inspect_df"]

print("\nInspeção manual para threshold:", threshold_to_inspect)
display(inspect_df[["source_index", "partido", "parlamentar", "n_chunks"]])

# escolha um discurso da tabela acima (source_index)
source_to_inspect = int(inspect_df.iloc[5]["source_index"])
row = inspect_df[inspect_df["source_index"] == source_to_inspect].iloc[0]

print("\nDiscurso selecionado:")
print("source_index:", row["source_index"])
print("partido:", row["partido"])
print("parlamentar:", row["parlamentar"])
print("n_chunks:", row["n_chunks"])

for i, chunk in enumerate(row["chunks"], start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)


Inspeção manual para threshold: 0.35


,source_index,partido,parlamentar,n_chunks
0,0,NOVO,Marcel Van Hattem,3
1,1,PCdoB,Renildo Calheiros,5
2,2,PT,Reginaldo Lopes,6
3,3,NOVO,Marcel Van Hattem,5
4,4,PL,Bia Kicis,20
5,6,PT,Reginaldo Lopes,19



Discurso selecionado:
source_index: 6
partido: PT
parlamentar: Reginaldo Lopes
n_chunks: 19

--- Chunk 1 ---
O SR.

--- Chunk 2 ---
REGINALDO LOPES (PT - MG.

--- Chunk 3 ---
Como Líder. Sem revisão do orador.) - Vou falar como Líder do PT, Presidenta Bia Kicis. Quero cumprimentar os Deputados. Tive a oportunidade de ouvir também a despedida do Deputado conterrâneo Tiago Mitraud, que fez um excelente trabalho. Deputado, quero parabenizar V.Exa. pela dedicação ao Estado brasileiro, ao povo brasileiro. Vamos sentir sua falta aqui.

--- Chunk 4 ---
Volte sempre.

--- Chunk 5 ---
Também o Deputado Paulo Ganime despediu-se aqui hoje.

--- Chunk 6 ---
Um foi candidato ao Governo do Rio, outro foi candidato a Vice-Presidente da República.

--- Chunk 7 ---
Então, quero aqui agradecer a convivência.

--- Chunk 8 ---
Quero também dizer que esse foi um ano importante, um ano de muito trabalho na Casa, um ano em que assumi a Liderança do meu partido.

--- Chunk 9 ---
Fizemos um planejamento no in

In [19]:

# ============================
# INSPEÇÃO MANUAL (ajuste aqui)
# ============================
threshold_to_inspect = 0.45

inspect_df = inspection_data[threshold_to_inspect]["inspect_df"]

print("\nInspeção manual para threshold:", threshold_to_inspect)
display(inspect_df[["source_index", "partido", "parlamentar", "n_chunks"]])

# escolha um discurso da tabela acima (source_index)
source_to_inspect = int(inspect_df.iloc[5]["source_index"])
row = inspect_df[inspect_df["source_index"] == source_to_inspect].iloc[0]

print("\nDiscurso selecionado:")
print("source_index:", row["source_index"])
print("partido:", row["partido"])
print("parlamentar:", row["parlamentar"])
print("n_chunks:", row["n_chunks"])

for i, chunk in enumerate(row["chunks"], start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)


Inspeção manual para threshold: 0.45


,source_index,partido,parlamentar,n_chunks
0,0,NOVO,Marcel Van Hattem,10
1,1,PCdoB,Renildo Calheiros,12
2,2,PT,Reginaldo Lopes,7
3,3,NOVO,Marcel Van Hattem,8
4,4,PL,Bia Kicis,26
5,6,PT,Reginaldo Lopes,28



Discurso selecionado:
source_index: 6
partido: PT
parlamentar: Reginaldo Lopes
n_chunks: 28

--- Chunk 1 ---
O SR.

--- Chunk 2 ---
REGINALDO LOPES (PT - MG.

--- Chunk 3 ---
Como Líder. Sem revisão do orador.) - Vou falar como Líder do PT, Presidenta Bia Kicis. Quero cumprimentar os Deputados. Tive a oportunidade de ouvir também a despedida do Deputado conterrâneo Tiago Mitraud, que fez um excelente trabalho. Deputado, quero parabenizar V.Exa.

--- Chunk 4 ---
pela dedicação ao Estado brasileiro, ao povo brasileiro.

--- Chunk 5 ---
Vamos sentir sua falta aqui.

--- Chunk 6 ---
Volte sempre.

--- Chunk 7 ---
Também o Deputado Paulo Ganime despediu-se aqui hoje.

--- Chunk 8 ---
Um foi candidato ao Governo do Rio, outro foi candidato a Vice-Presidente da República.

--- Chunk 9 ---
Então, quero aqui agradecer a convivência.

--- Chunk 10 ---
Quero também dizer que esse foi um ano importante, um ano de muito trabalho na Casa, um ano em que assumi a Liderança do meu partido.

--- Chunk 

## Execução da geração de embeddings

In [ ]:
from pathlib import Path

# antes da eleição
csv_path_antes = Path.cwd().parent / "data" / "running_files" / "political_discourses_ini_02072022_fim_29102022.csv"

df_csv_antes = pd.read_csv(csv_path_antes)

In [ ]:
emb_df_test, emb_matrix_test, _ = generate_discourse_embeddings(
        dataframe=df_csv_antes,
        party="UNIÃO",
        text_col="preprocess_disc",
        similarity_threshold=0.35,
        min_sentences_per_chunk=1,
        max_sentences_per_chunk=None,
        save_files=True,
        source_csv_name="political_discourses_ini_02072022_fim_29102022.csv"
)

...................................................
... Função generate_discourse_embeddings iniciada ...


c:\Users\luizf\github\luizbrandi\political-discourse-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2104.63it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... Total de chunks gerados: 117 ...


Batches: 100%|██████████| 4/4 [00:04<00:00,  1.20s/it]

... Arquivo CSV salvo em: data/running_files/embeddings\discourses_embeddings_UNIÃO_ini_02072022_fim_29102022.csv ...
... Arquivo NPY salvo em: data/running_files/embeddings\discourses_embeddings_UNIÃO_ini_02072022_fim_29102022.npy ...
... Função generate_discourse_embeddings encerrada ...
....................................................


In [6]:
# depois da eleição
csv_path_depois = Path.cwd().parent / "data" / "running_files" / "political_discourses_ini_31102022_fim_02012023.csv"
df_csv_depois = pd.read_csv(csv_path_depois)

emb_df_test, emb_matrix_test, _ = generate_discourse_embeddings(
        dataframe=df_csv_depois,
        party="UNIÃO",
        text_col="preprocess_disc",
        similarity_threshold=0.35,
        min_sentences_per_chunk=1,
        max_sentences_per_chunk=None,
        save_files=True,
        source_csv_name="political_discourses_ini_31102022_fim_02012023.csv"
)

...................................................
... Função generate_discourse_embeddings iniciada ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1798.42it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... Total de chunks gerados: 103 ...


Batches: 100%|██████████| 4/4 [00:04<00:00,  1.17s/it]


... Arquivo CSV salvo em: data/running_files/embeddings\discourses_embeddings_UNIÃO_ini_31102022_fim_02012023.csv ...
... Arquivo NPY salvo em: data/running_files/embeddings\discourses_embeddings_UNIÃO_ini_31102022_fim_02012023.npy ...
... Função generate_discourse_embeddings encerrada ...
....................................................
